In [ ]:
import os
from utilities.extractframes import FrameExtractor
import graphene_demoA 
import graphene_demoB 
from llama_ask import run_llama
directory_path = 'eval/quant'


Observer A scene understanding: 

In [ ]:
def run_oic():
    output_dir = "out"
    fps = 2
    window_size = 2
    for video in os.scandir(directory_path):
        if video.is_file():
            if not video.name.startswith('.'):
                ex = FrameExtractor(directory_path+'/'+video.name, fps_to_save=fps, window_size=window_size)
                fps_to_save = ex.main()
                
                # instantiate graphene "OIC core that runs RelTR"  
                g = graphene_demoA.Graphene(alpha=0.7, min_assignment_conf=0.3)
                
                # check if the out directory exists else create one
                if not os.path.isdir(output_dir):
                    os.mkdir(output_dir)
                  
                # prepare output files 
                text = os.path.splitext(video.name)[0]+'graph2text.txt'
                img_dir_path = directory_path+'/'+os.path.splitext(video.name)[0]+'-opencv'
    
                  
                # classify images from the image directory
                #g.classify_images(image_path=img_dir_path) # without windowing
                
                g.classify_images_window(image_path=img_dir_path, fps_to_save=fps_to_save, window_size=window_size)
                
                # generate relationship graph
                graph_dir_path = img_dir_path + "/img/JSON"
                
                g.generate_temporal_graph_frames(graph_dir_path, img_dir_path + "/img")
                
                # save textual output in the out directory
                g.tg.to_text(os.path.join(output_dir, text), fps_to_save=fps_to_save, window_size=window_size)
      
                if os.path.isfile(os.path.join(output_dir,text)):
                    with open(os.path.join(output_dir,text)) as f:
                        prompt_content = "".join(map(str, f.readlines()))
                        return str(prompt_content)
                else:
                    continue

In [ ]:
if os.path.isdir(directory_path):
    prompt_A = run_oic()
    print("CUIDs from DemoA".format(prompt_A))

Common knowledge base to publish profile of the observers

In [ ]:
ObserverA = {
    "type": True,
    "color_perception": True,
    "size_perception": True,
    "depth_perception": False,
    "relationship": True,
    "position": True
} 

ObserverB = {
    "type": True,
    "color_perception": True,
    "size_perception": True,
    "depth_perception": False,
    "relationship": False,
    "position": True
} 

Empathize with Observer B 

In [ ]:
car_9ae5 = {'type': 'car', 'position': '[Xmin = 179.9986572265625, Ymin = 186.359130859375, Xmax = 205.7042236328125, Ymax = 216.1868896484375]', 'relationship': 'car has wheel'}
id = car_9ae5

items = {k for k in (set(id.keys()) & set(ObserverB.keys())) if id[k] and ObserverB[k]}

perceived_set = {}

for i in items:
    perceived_set.update({i:id.get(i)})

print(perceived_set)

Ask a question to Observer B

In [ ]:
cuid = perceived_set
question = f"Can you find the {cuid=}"


print(question)

Observer B perceives the scene 

In [ ]:
def run_yolo():
    output_dir = "out"
    fps = 2
    window_size = 2
    for video in os.scandir(directory_path):
        if video.is_file():
            if not video.name.startswith('.'):
                ex = FrameExtractor(directory_path+'/'+video.name, fps_to_save=fps, window_size=window_size)
                fps_to_save = ex.main()
                
                # instantiate graphene "OIC core that runs yolo with alpha=0 making relationships count 0 and 1 on spatial similarity"  
                g = graphene_demoB.Graphene(alpha=0, min_assignment_conf=0.6)
                
                # check if the out directory exists else create one
                if not os.path.isdir(output_dir):
                    os.mkdir(output_dir)
                  
                # prepare output files 
                text = os.path.splitext(video.name)[0]+'graph2text.txt'
                img_dir_path = directory_path+'/'+os.path.splitext(video.name)[0]+'-opencv'
    
                  
                # classify images from the image directory
                #g.classify_images(image_path=img_dir_path) # without windowing
                
                g.classify_images_window(image_path=img_dir_path, fps_to_save=fps_to_save, window_size=window_size)
                
                # generate relationship graph
                graph_dir_path = img_dir_path + "/img/JSON"
                
                g.generate_temporal_graph_frames(graph_dir_path, img_dir_path + "/img")
                
                # save textual output in the out directory
                g.tg.to_text(os.path.join(output_dir, text), fps_to_save=fps_to_save, window_size=window_size)
      
                if os.path.isfile(os.path.join(output_dir,text)):
                    with open(os.path.join(output_dir,text)) as f:
                        prompt_content = "".join(map(str, f.readlines()))
                        return str(prompt_content)
                else:
                    continue

In [ ]:
if os.path.isdir(directory_path):
  promptB : str = run_yolo()
  print("CUIDs from ObsB".format(promptB))

        Compare the identities to understand which to which object the question refers to and answer the question with confidence in the matching identity of the object detected

In [ ]:
delta = 20

from llama_ask import run_llama

answer = run_llama(promptB, f"{question} with matching tolerance of {delta} percent? If yes, tell me some fun facts about the object")

print(answer)

In [ ]:
answer = run_llama(promptB, f"{question} with matching tolerance of {delta} percent? If yes, tell me some fun facts about the object")

print(answer)